Mandatory imports for our Data Cleaning
Read .csv dataset into pandas DataFrame

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [ ]:
# Adjust display options to see all columns
pd.set_option('display.max_columns', None)

# Read dataset from .csv file
df = pd.read_csv('../data/raw/bkk_positions.csv', dtype=str)

print(f'Initial dataset shape: {df.shape}')
df.head()

Initial dataset shape: (211765, 18)


,ingested_at,id,vehicle.vehicle.id,vehicle.vehicle.label,vehicle.vehicle.licensePlate,vehicle.position.bearing,vehicle.position.latitude,vehicle.position.longitude,vehicle.timestamp,vehicle.currentStatus,vehicle.position.speed,vehicle.trip.tripId,vehicle.trip.routeId,vehicle.trip.startDate,vehicle.trip.scheduleRelationship,vehicle.stopId,vehicle.currentStopSequence,vehicle.congestionLevel
0,2026-04-16 19:05:09.432211,VehiclePosition-BKK_1,1,Deák F. tér M (City centre),TSM002,218.0,47.432663,19.261341,1776333261,IN_TRANSIT_TO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-16 19:05:09.438127,VehiclePosition-BKK_100,100,NaN,AADI603,72.0,47.47466,19.02789,1776360439,IN_TRANSIT_TO,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-16 19:05:09.441053,VehiclePosition-BKK_1000,1000,"Csepel, Soroksári rév",AOJM876,316.0,47.456635,19.116358,1776366299,IN_TRANSIT_TO,0.0,D12367625,1480,20260416,SCHEDULED,F01370,11.0,NaN
3,2026-04-16 19:05:09.444185,VehiclePosition-BKK_1003,1003,Kőbánya-Kispest M,AOJM877,289.0,47.422478,19.069391,1776366302,IN_TRANSIT_TO,3.601108,D12367595,1480,20260416,SCHEDULED,F04185,13.0,NaN
4,2026-04-16 19:05:09.447917,VehiclePosition-BKK_1005,1005,NaN,AOJM879,266.0,47.443203,19.082645,1776324825,IN_TRANSIT_TO,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df['latitude'] = pd.to_numeric(df['vehicle.position.latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['vehicle.position.longitude'], errors='coerce')

df = df.dropna(subset=['latitude', 'longitude'])

df = df[(df['latitude'] != 0.0) & (df['longitude'] != 0.0)]

valid_statuses = ['IN_TRANSIT_TO', 'STOPPED_AT', 'INCOMING_AT']

df = df[df['vehicle.currentStatus'].isin(valid_statuses)]

print("Cleaned Vehicle Status Distribution:")
print(df['vehicle.currentStatus'].value_counts())
print(f"Final dataset shape before saving: {df.shape}")


Cleaned Vehicle Status Distribution:
vehicle.currentStatus
IN_TRANSIT_TO    96642
STOPPED_AT       27350
Name: count, dtype: int64
Final dataset shape before saving: (123992, 20)


In [ ]:
# Drop unnecessary raw columns to save space
cols_to_drop = ['vehicle.position.latitude', 'vehicle.position.longitude', 'delay'] # Add others as needed
df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Save the cleaned dataset
processed_path = '../data/processed/bkk_cleaned.csv'
df_clean.to_csv(processed_path, index=False)

print(f"Cleaned dataset saved to {processed_path} with shape {df_clean.shape}")

Cleaned dataset saved to ../data/processed/bkk_cleaned.csv with shape (123992, 18)


In [ ]:
import ast
import re
import pandas as pd
import numpy as np

print("Loading Trip Updates (ETA Data)...")
df_delays = pd.read_csv('../data/raw/bkk_delays.csv', dtype=str)

if 'tripUpdate.trip.tripId' in df_delays.columns:
    df_delays = df_delays.rename(columns={'tripUpdate.trip.tripId': 'vehicle.trip.tripId'})

def safe_parse_list(val):
    if pd.isna(val):
        return []
    try:
        clean_val = re.sub(r'\bnan\b', 'None', str(val))
        return ast.literal_eval(clean_val)
    except Exception:
        return []

print("Unpacking nested arrival arrays...")
df_delays['tripUpdate.stopTimeUpdate'] = df_delays['tripUpdate.stopTimeUpdate'].apply(safe_parse_list)

# Explode the stops
df_delays_exploded = df_delays.explode('tripUpdate.stopTimeUpdate')

# NEW: Extract the absolute arrival TIME instead of delay
def extract_arrival_time(stop_dict):
    if isinstance(stop_dict, dict):
        arrival = stop_dict.get('arrival', {})
        if isinstance(arrival, dict):
            return arrival.get('time', np.nan)
    return np.nan

df_delays_exploded['arrival_timestamp'] = df_delays_exploded['tripUpdate.stopTimeUpdate'].apply(extract_arrival_time)

# We only want the immediate NEXT stop for the bus. 
# Sort by the timestamp and keep only the first upcoming stop for each trip.
df_delays_exploded = df_delays_exploded.dropna(subset=['arrival_timestamp'])
df_delays_exploded['arrival_timestamp'] = df_delays_exploded['arrival_timestamp'].astype(float)
df_delays_exploded = df_delays_exploded.sort_values('arrival_timestamp')

# Keep only the closest upcoming ETA for each bus
eta_features = df_delays_exploded[['vehicle.trip.tripId', 'arrival_timestamp']].drop_duplicates(subset=['vehicle.trip.tripId'])

print(f"ETA features ready for merging: {eta_features.shape[0]} unique trips.")

# ---------------------------------------------------------
# THE GRAND MERGE
# ---------------------------------------------------------
print("Merging physical positions with ETAs...")

# df is your cleaned bkk_positions dataset
df = df.merge(eta_features, on='vehicle.trip.tripId', how='left')

df = df.dropna(subset=['arrival_timestamp'])

# ---------------------------------------------------------
# CREATE THE TARGET VARIABLE
# ---------------------------------------------------------
# Ensure timestamps are numeric
df['vehicle.timestamp'] = pd.to_numeric(df['vehicle.timestamp'], errors='coerce')

# Target Variable: How many seconds from the GPS ping until it hits the stop?
df['seconds_to_next_stop'] = df['arrival_timestamp'] - df['vehicle.timestamp']

# Filter out impossible physics (e.g., negative time or extreme outliers > 1 hour)
df = df[(df['seconds_to_next_stop'] > 0) & (df['seconds_to_next_stop'] < 3600)]

df.to_csv('../data/processed/bkk_cleaned.csv', index=False)

print(f"FINAL DATASET READY FOR ML: {df.shape}")
print(df[['vehicle.trip.tripId', 'vehicle.timestamp', 'arrival_timestamp', 'seconds_to_next_stop']].head())

Loading Trip Updates (ETA Data)...
Unpacking nested arrival arrays...
ETA features ready for merging: 5973 unique trips.
Merging physical positions with ETAs...
FINAL DATASET READY FOR ML: (550, 22)
      vehicle.trip.tripId  vehicle.timestamp  arrival_timestamp  \
80404         D1238115938         1776615546       1.776618e+09   
81880         D1238115938         1776615837       1.776618e+09   
82110           D12995304         1776615845       1.776616e+09   
83357         D1238115938         1776616142       1.776618e+09   
83367          D106182597         1776616137       1.776617e+09   

       seconds_to_next_stop  
80404                2093.0  
81880                1802.0  
82110                 137.0  
83357                1497.0  
83367                 420.0  
